<a href="https://colab.research.google.com/github/nimishac28/AI-agent-Transaction/blob/main/AI_agent_gmail_tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
%pip install -q langchain langchain-openai pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
%pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [26]:
import re

SENSITIVE_PATTERNS = [
    r"\botp\b",
    r"\bone[-\s]?time\s+password\b",
    r"\bverification\s+code\b",
    r"\bsecurity\s+code\b",
    r"\bpassword\b",
    r"\bpasscode\b",
    r"\bauthentication\s+code\b",
    r"\blogin\s+code\b",
    r"\bprofile\s+summary\b",
    r"\bagreement\b",
    r"\bauth\w*\b",
    r"\bverification\w*\b",]

SUCCESSFUL_TRANSACTION_PATTERNS = [
    r"\btransaction\b",
    r"\btransaction\s+(?:was\s+)?successful\b",
    r"\bpayment\s+(?:was\s+)?successful\b",
    r"\bcard\s+(?:was\s+)?used\b",
    r"\bwas\s+used\b",
    r"\bhas\s+(?:just\s+)?been\s+swiped\b",
    r"\bswiped\b",
    r"\bspent\b",
    r"\bpurchase\b",
    r"\bcharged\b",
    r"\bpayment\s+was\s+made\b",
]

NON_TRANSACTION_PATTERNS = [
    # Loans and credit offers
    r"\bloan\b",
    r"\bloan\s+(?:amount|offer|update|available)\b",
    r"\bcredit\s+limit\b",

    # Statements and bills
    r"\b(?:credit\s+card\s+)?statement\b",
    r"\bbill\b",
    r"\bbill\s+payment\b",
    r"\bminimum\s+(?:amount\s+)?due\b",
    r"\bpayment\s+due\b",

    # Marketing / promotional emails
    r"\boffer(?:s)?\b",
    r"\bpromotion(?:al)?\b",
    r"\bpre[-\s]?approved\b",
]

def is_sensitive_email(subject: str, body: str) -> bool:
    """
    Returns True if the email contains authentication credentials
    or other sensitive security-related information.
    """

    text = f"{subject} {body}"

    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in SENSITIVE_PATTERNS
    )


user_cards = input(
    "Enter the banks/cards you want to track (comma-separated): "
)

TRACKED_CARDS = [
    card.strip().lower()
    for card in user_cards.split(",")
]

print("Tracking:", TRACKED_CARDS)

def build_card_keywords(TRACKED_CARDS: str):
    """
    Create common keyword variations for a tracked card.
    """
    card = TRACKED_CARDS.strip().lower()

    return [
        card,
        f"{card} credit card"
    ]


CARD_KEYWORDS = {
    card: build_card_keywords(card)
    for card in TRACKED_CARDS
}


def get_tracked_card(subject: str, body: str):
    text = f"{subject} {body}".lower()

    for card_name, config in TRACKED_CARDS.items():
        for keyword in config["keywords"]:
            if keyword.lower() in text:
                return card_name

    return None

def is_tracked_card_email(subject: str, body: str) -> bool:
    """
    Returns True only if the email belongs to one of the
    user's configured cards/banks.
    """

    text = f"{subject} {body}".lower()

      #  First reject known non-purchase emails
    if any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in NON_TRANSACTION_PATTERNS
    ):
        return False

    #Then look for actual spending activity
    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in SUCCESSFUL_TRANSACTION_PATTERNS
    )

def is_successful_transaction(subject: str, body: str) -> bool:
    text = f"{subject} {body}"

    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in SUCCESSFUL_TRANSACTION_PATTERNS
    )
OTP_CODE = r"\b(?:otp|one[-\s]?time\s+password|verification\s+code)\b\D{0,40}\b\d{4,8}\b"

def should_process_transaction_email(subject: str, body: str) -> bool:
    # 1. Block emails containing an actual OTP code (e.g. "OTP is 482913")
    if re.search(OTP_CODE, body, re.IGNORECASE):
        return False

    # 2. Sensitive / non-transaction checks on SUBJECT only (footers trigger false alarms)
    if is_sensitive_email(subject, ""):
        return False
    if any(re.search(p, subject, re.IGNORECASE) for p in NON_TRANSACTION_PATTERNS):
        return False

    # 3. Success check can use subject + body
    return is_successful_transaction(subject, body)


from pydantic import BaseModel, Field
from typing import Optional


class CreditCardTransaction(BaseModel):
    bank_name: Optional[str] = Field(
        default=None,
        description="Bank or credit card issuer mentioned in the email"
    )

    card_last_four: Optional[str] = Field(
        default=None,
        description="Last four digits of the credit card, if explicitly mentioned"
    )

    amount: Optional[float] = Field(
        default=None,
        description="Transaction amount as a number without currency symbols"
    )

    merchant: Optional[str] = Field(
        default=None,
        description="Merchant where the card was used"
    )

    transaction_date: Optional[str] = Field(
        default=None,
        description="Transaction date if explicitly mentioned in the email"
    )




Tracking: ['hdfc']


In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

structured_llm = llm.with_structured_output(CreditCardTransaction)

In [28]:
def extract_transaction(subject: str, body: str):

    # 🛡️ Final safety check
    if not should_process_transaction_email(subject, body):
        return {
            "status": "IGNORED",
            "reason": "Email did not pass the transaction security filter"
        }

    message = f"""
Extract the credit card transaction details from this email.

Rules:
- Extract only information explicitly stated.
- Do not guess or invent missing information.
- Amount must be returned as a number without currency symbols.
- If a date is not explicitly present, return None.
- Extract only the last four card digits, never a complete card number.

Subject:
{subject}

Email body:
{body}
"""

    transaction = structured_llm.invoke(message)

    return {
        "status": "PROCESSED",
        "transaction": transaction
    }

In [29]:
import os

json_files = [f for f in os.listdir() if f.endswith(".json")]

print(json_files)

from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly"
]

from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

creds = None
if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
        except Exception:
            creds = None
    if not creds:
        flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
        creds = flow.run_local_server(port=0)
    with open("token.json", "w") as f:
        f.write(creds.to_json())


gmail_service = build(
    "gmail",
    "v1",
    credentials=creds
)

def build_gmail_query(tracked_cards):
    """
    Build a Gmail search query that looks only for emails
    related to the user's tracked cards.
    """
    return " OR ".join(tracked_cards)


gmail_query = build_gmail_query(TRACKED_CARDS)

print("Searching for:", gmail_query)

results = gmail_service.users().messages().list(
    userId="me",
    q=gmail_query,
    maxResults=20
).execute()

messages = results.get("messages", [])

print(f"Found {len(messages)} matching emails")

def get_email_metadata(message_id):
    message = gmail_service.users().messages().get(
        userId="me",
        id=message_id,
        format="metadata",
        metadataHeaders=["Subject", "From", "Date"]
    ).execute()

    headers = {
        header["name"]: header["value"]
        for header in message.get("payload", {}).get("headers", [])
    }

    return {
        "id": message_id,
        "subject": headers.get("Subject", ""),
        "from": headers.get("From", ""),
        "date": headers.get("Date", "")
    }
for message in messages:
    email = get_email_metadata(message["id"])

    subject = email["subject"]

    # Test transaction classification using subject only for now
    is_transaction = should_process_transaction_email(subject, "")

    if is_transaction:
        status = "✅ PROCESS"
    else:
        status = "⏭️ IGNORE"

    print(f"{status} | {subject}")



['credentials.json', 'token.json']
Searching for: hdfc
Found 8 matching emails
✅ PROCESS | Fwd: A payment was made using your Credit Card
✅ PROCESS | Fwd: A payment was made using your Credit Card
✅ PROCESS | Fwd: A payment was made using your Credit Card
✅ PROCESS | Fwd: A payment was made using your Credit Card
✅ PROCESS | Fwd: A payment was made using your Credit Card
✅ PROCESS | Fwd: A payment was made using your Credit Card
⏭️ IGNORE | Fwd: Credit Card xx3353: Higher Loan Amount Update
⏭️ IGNORE | Fwd: your credit card bill payment was successful


In [30]:
import base64

def get_email_body(message_id):
    msg = gmail_service.users().messages().get(userId="me", id=message_id, format="full").execute()

    def walk(part):
        mime = part.get("mimeType", "")
        data = part.get("body", {}).get("data")
        if data and mime in ("text/plain", "text/html"):
            text = base64.urlsafe_b64decode(data).decode("utf-8", errors="ignore")
            return re.sub(r"<[^>]+>", " ", text) if mime == "text/html" else text
        for p in part.get("parts", []) or []:
            found = walk(p)
            if found:
                return found
        return ""

    return walk(msg.get("payload", {})) or msg.get("snippet", "")

In [31]:
for m in messages:
    meta = get_email_metadata(m["id"])
    body = get_email_body(m["id"])
    text = f"{meta['subject']} {body}"
    print(meta["subject"][:60],
          "| sensitive:", is_sensitive_email(meta["subject"], body),
          "| non-txn:", any(re.search(p, text, re.I) for p in NON_TRANSACTION_PATTERNS),
          "| success:", is_successful_transaction(meta["subject"], body))

Fwd: A payment was made using your Credit Card | sensitive: True | non-txn: True | success: True
Fwd: A payment was made using your Credit Card | sensitive: True | non-txn: True | success: True
Fwd: A payment was made using your Credit Card | sensitive: True | non-txn: True | success: True
Fwd: A payment was made using your Credit Card | sensitive: True | non-txn: False | success: True
Fwd: A payment was made using your Credit Card | sensitive: True | non-txn: True | success: True
Fwd: A payment was made using your Credit Card | sensitive: True | non-txn: True | success: True
Fwd: Credit Card xx3353: Higher Loan Amount Update | sensitive: False | non-txn: True | success: False
Fwd: your credit card bill payment was successful | sensitive: False | non-txn: True | success: True


In [32]:
# Cell 9: extract transactions into a table
import pandas as pd

transactions = []
for m in messages:
    meta = get_email_metadata(m["id"])
    body = get_email_body(m["id"])
    result = extract_transaction(meta["subject"], body)
    if result["status"] == "PROCESSED":
        row = result["transaction"].model_dump()
        row["email_date"] = meta["date"]
        transactions.append(row)

df = pd.DataFrame(transactions)
df

,bank_name,card_last_four,amount,merchant,transaction_date,email_date
0,HDFC Bank,3353,3150.00,"WLI Toni And Guy, Gold","27 Sep, 2026 at 15:31:39","Sat, 14 Nov 2026 21:26:27 +0530"
1,HDFC Bank,3353,8000.00,PRAKASH HOSPITAL,"03 Oct, 2026 at 16:54:49","Sat, 14 Nov 2026 21:26:18 +0530"
2,HDFC Bank,3353,3319.09,FIRSTCRYBABY,"12 Oct, 2026 at 13:52:16","Sat, 14 Nov 2026 21:26:08 +0530"
3,HDFC Bank,3353,1500.00,IND Noida Power Compan,"20 Oct, 2026 at 12:30:02","Sat, 14 Nov 2026 21:25:59 +0530"
4,HDFC Bank,3353,4841.60,ASSPL,"21 Oct, 2026 at 23:12:28","Sat, 14 Nov 2026 21:25:42 +0530"
5,HDFC Bank,3353,2626.00,ASSPL,"25 Oct, 2026 at 19:09:41","Sat, 14 Nov 2026 21:25:32 +0530"


In [ ]:
# Cell 10: ask questions and get summaries

def ask_transactions(question: str) -> str:
    if df.empty:
        return "No transactions found."
    prompt = f"""You are a personal finance assistant.
Answer using ONLY the transactions below. If the answer isn't in the data, say you don't know.

Transactions (CSV):
{df.to_csv(index=False)}

Question: {question}"""
    return llm.invoke(prompt).content

print(ask_transactions("What did I spend at each merchant?"))
print(ask_transactions("Summarise my transactions this month."))

# Exact totals: let pandas do the maths, not the LLM
if not df.empty:
    print(df.groupby("bank_name")["amount"].sum())

Here is the amount you spent at each merchant based on the transactions:

- WLI Toni And Guy, Gold: 3150.0
- PRAKASH HOSPITAL: 8000.0
- FIRSTCRYBABY: 3319.09
- IND Noida Power Compan: 1500.0
- ASSPL: 4841.6 + 2626.0 = 7467.6
For November 2026, there are no transactions recorded in the provided data.
bank_name
HDFC Bank    23436.69
Name: amount, dtype: float64
